This notebook just has the creativity metric definitions outside of the demo for ease of use

This first cell has the original creativity metric used in the first demo and the presentation

In [ ]:
from typing import Union

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

from sklearn.neighbors import NearestNeighbors

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)

    def probs(self, x):
        return F.softmax(self.forward(x), dim=1)
# ─────────────────────────────────────────────────────────────
# 5. Creativity scoring  (Ramaswamy et al.)
# ─────────────────────────────────────────────────────────────

def score_creativity(images: torch.Tensor, clf: Classifier,
                     train_flat: np.ndarray, device: Union[str, torch.device] = "cpu",
                     alpha: float = 0.5):
    """
    Compute per-image creativity scores.

    Creativity(x) = Novelty(x)^alpha * Usefulness(x)^(1-alpha)

    Usefulness  = min(P(2|x), P(6|x))
                  Peaks when the classifier is equally confident the image
                  is a 2 AND a 6 — the ideal creative combination.

    Novelty     = nn_dist / (1 + nn_dist)
                  Saturates toward 1 as the image moves away from all
                  training examples; 0 if the image is a copy of one.
    """
    clf.eval()
    with torch.no_grad():
        probs = clf.probs(images.to(device)).cpu().numpy()   # (N, 2)

    usefulness = np.minimum(probs[:, 0], probs[:, 1])        # (N,)

    flat = images.view(images.size(0), -1).cpu().numpy()     # (N, 784)
    nbrs = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(train_flat)
    nn_dist = nbrs.kneighbors(flat)[0][:, 0]                 # (N,)
    novelty = nn_dist / (1.0 + nn_dist)

    creativity = (novelty ** alpha) * (usefulness ** (1.0 - alpha))
    return creativity, novelty, usefulness

This generates the following image: ![](./images/creative_optimised.png)

This next snippet is a more sophisticated algorithm that uses shape features to attempt to judge creativity and task accuracy

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5b. Extended creativity components
# ─────────────────────────────────────────────────────────────

# ── 5b-i. General autoencoder (trained on all MNIST) ─────────

class GeneralAE(nn.Module):
    """Simple AE trained on all 10 MNIST classes.  Low reconstruction
    error means the image looks like a plausible handwritten mark."""
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_general_ae(ae: GeneralAE, epochs: int = 10, lr: float = 1e-3,
                     device: Union[str, torch.device] = "cpu"):
    """Train the general AE on all 10 MNIST classes."""
    full_dataset = torchvision.datasets.MNIST(
        root="./data", train=True, download=True,
        transform=transforms.ToTensor()
    )
    loader = DataLoader(full_dataset, batch_size=256, shuffle=True)
    opt = optim.Adam(ae.parameters(), lr=lr)
    ae.train()
    for epoch in range(epochs):
        total = 0.0
        for x, _ in loader:
            x = x.to(device)
            loss = F.mse_loss(ae(x), x)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"  [AE] epoch {epoch+1:>3}/{epochs}  loss={total/len(loader):.6f}")


# ── 5b-ii. Persistent homology ────────────────────────────────
# Uses gudhi (cubical complex, H1) + persim (Wasserstein distance).

try:
    import gudhi
    from persim import wasserstein as _wasserstein
    TOPO_AVAILABLE = True
except ImportError:
    TOPO_AVAILABLE = False
    print("gudhi / persim not found — structural novelty will be set to 0.")
    print("Install with:  pip install gudhi persim")


def _persistence_h1(img_np: np.ndarray) -> np.ndarray:
    """H1 persistence diagram via cubical sublevel-set filtration.
    Infinite deaths are dropped; returns [[0,0]] if no finite H1 features."""
    cc = gudhi.CubicalComplex(
        dimensions=[28, 28],
        top_dimensional_cells=img_np.flatten().tolist()
    )
    cc.compute_persistence()
    dgm = cc.persistence_intervals_in_dimension(1)
    if len(dgm) == 0:
        return np.array([[0.0, 0.0]])
    finite = dgm[np.isfinite(dgm[:, 1])]
    return finite if len(finite) > 0 else np.array([[0.0, 0.0]])


def compute_train_diagrams(train_flat: np.ndarray,
                            n_samples: int = 150,
                            seed: int = 0) -> list:
    """Compute H1 diagrams for a random subset of training images."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(train_flat), size=min(n_samples, len(train_flat)),
                     replace=False)
    diagrams = []
    for i, j in enumerate(idx):
        diagrams.append(_persistence_h1(train_flat[j].reshape(28, 28)))
        if (i + 1) % 50 == 0:
            print(f"  [TOPO] {i+1}/{len(idx)} training diagrams computed")
    return diagrams

def _wd(d1, d2) -> float:
    r = _wasserstein(d1, d2, matching=False)
    return float(r[0]) if isinstance(r, tuple) else float(r)

def topological_novelty(images: torch.Tensor, train_diagrams: list) -> np.ndarray:
    """Min Wasserstein-1 distance to any training H1 diagram, normalised
    as d / (1 + d).  Requires TOPO_AVAILABLE."""
    imgs_np = images.squeeze().numpy()   # (N, 28, 28)
    scores = []
    for img in imgs_np:
        dgm = _persistence_h1(img)
        min_d = min(_wd(dgm, td) for td in train_diagrams)
        scores.append(min_d / (1.0 + min_d))
    return np.array(scores, dtype=np.float32)


# ── 5b-iii. Jensen-Shannon ambiguity ─────────────────────────


def js_ambiguity(probs: np.ndarray) -> np.ndarray:
    """
    JS divergence (log base 2) between each row of `probs` and the
    uniform distribution u = [0.5, 0.5].  Result is in [0, 1];
    equals 1 when p = [0.5, 0.5] (maximum uncertainty).
    """
    u = np.full_like(probs, 0.5)
    m = 0.5 * (probs + u)
    eps = 1e-10
    kl_pm = np.sum(probs * np.log2((probs + eps) / (m + eps)), axis=1)
    kl_um = np.sum(u     * np.log2((u     + eps) / (m + eps)), axis=1)
    return np.clip(0.5 * kl_pm + 0.5 * kl_um, 0.0, 1.0)


def js_ambiguity_torch(probs: torch.Tensor) -> torch.Tensor:
    """Differentiable version of js_ambiguity for gradient-based optimisation."""
    u = torch.full_like(probs, 0.5)
    m = 0.5 * (probs + u)
    eps = 1e-10
    kl_pm = (probs * torch.log2((probs + eps) / (m + eps))).sum(dim=1)
    kl_um = (u     * torch.log2((u     + eps) / (m + eps))).sum(dim=1)
    return torch.clamp(0.5 * kl_pm + 0.5 * kl_um, 0.0, 1.0)


# ── 5b-iv. Extended creativity score ─────────────────────────

def score_creativity_extended(images: torch.Tensor,
                               clf: Classifier,
                               ae: GeneralAE,
                               train_flat: np.ndarray,
                               train_diagrams: list,
                               device: Union[str, torch.device] = "cpu",
                               weights: tuple = (1/3, 1/3, 1/3)):
    """
    ExtCreativity = StructuralNovelty^w1 · Realness^w2 · Ambiguity^w3

    StructuralNovelty — min Wasserstein-1 distance in H1 diagram space,
                        normalised by d/(1+d).  0 if gudhi unavailable.
    Realness          — 1 / (1 + MSE(x, AE(x))); near 1 for digit-like images.
    Ambiguity         — JS divergence from classifier output to uniform [0.5, 0.5];
                        equals 1 when the classifier is maximally uncertain.
    """
    w1, w2, w3 = weights
    ae.eval()
    clf.eval()

    with torch.no_grad():
        imgs_dev = images.to(device)
        recon    = ae(imgs_dev)
        mse      = F.mse_loss(recon, imgs_dev, reduction="none") \
                     .mean(dim=[1, 2, 3]).cpu().numpy()
        probs    = clf.probs(imgs_dev).cpu().numpy()

    realness  = 1.0 / (1.0 + mse)
    ambiguity = js_ambiguity(probs)

    if TOPO_AVAILABLE and len(train_diagrams) > 0:
        struct_novelty = topological_novelty(images.cpu(), train_diagrams)
    else:
        struct_novelty = np.zeros(len(images), dtype=np.float32)

    ext_creativity = (struct_novelty ** w1) * (realness ** w2) * (ambiguity ** w3)
    return ext_creativity, struct_novelty, realness, ambiguity

This generates the following images ![](./images/creative_ext_optimised.png)